In [1]:
!pip install rdflib

In [8]:
!pip uninstall -y numpy karateclub
!pip install numpy==1.23.5 karateclub


Found existing installation: numpy 1.22.4
Uninstalling numpy-1.22.4:
  Successfully uninstalled numpy-1.22.4
Found existing installation: karateclub 1.3.3
Uninstalling karateclub-1.3.3:
  Successfully uninstalled karateclub-1.3.3
  Using cached karateclub-1.3.3-py3-none-any.whl
INFO: pip is looking at multiple versions of karateclub to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 86.9 MB/s eta 0:00:00
  Created wheel for karateclub: filename=karateclub-1.3.0-py3-none-any.whl size=98558 sha256=d04ca9b3e85ee52a58e7a6991269edd423d7206

In [2]:
!pip install karateclub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 47.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 81.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 k

In [4]:
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, XSD
from datetime import datetime, timedelta
import random
import hashlib

# Namespaces
ex = Namespace("http://example.org/")
health = Namespace("http://example.org/health/")
finance = Namespace("http://example.org/finance/")
general = Namespace("http://example.org/general/")
time = Namespace("http://example.org/time/")

g = Graph()
g.bind("ex", ex)
g.bind("health", health)
g.bind("finance", finance)
g.bind("general", general)
g.bind("time", time)

def generate_stmt_id(s, p, o):
    raw = f"{str(s)}_{str(p)}_{str(o)}"
    return URIRef(f"http://example.org/event/{hashlib.md5(raw.encode()).hexdigest()}")

def add_temporal_triple(g, s, p, o, timestamp):
    g.add((s, p, o))
    stmt_id = generate_stmt_id(s, p, o)
    g.add((stmt_id, time.timestamp, Literal(timestamp.isoformat(), datatype=XSD.dateTime)))

# --- Template-driven Event Simulations ---
def simulate_doctor_visit(g, person, doctor, location, symptoms, diagnoses, start_date):
    event_time = start_date + timedelta(days=random.randint(0, 5))
    for symptom in symptoms:
        add_temporal_triple(g, person, health.hasSymptom, health[symptom], event_time)
    for diagnosis in diagnoses:
        add_temporal_triple(g, person, health.diagnosedWith, health[diagnosis], event_time)
    add_temporal_triple(g, person, health.visitedDoctor, health[doctor], event_time)
    add_temporal_triple(g, person, health.atHospital, health[location], event_time)

def simulate_finance_event(g, person, bank, asset, amount, start_date):
    event_time = start_date + timedelta(days=random.randint(0, 5))
    add_temporal_triple(g, person, finance.investedIn, finance[asset], event_time)
    add_temporal_triple(g, person, finance.accountAt, finance[bank], event_time)
    add_temporal_triple(g, person, finance.transactionAmount, Literal(amount, datatype=XSD.float), event_time)

def simulate_routine_activity(g, person, activity, location, start_date):
    event_time = start_date + timedelta(days=random.randint(0, 5))
    add_temporal_triple(g, person, general.participatedIn, general[activity], event_time)
    add_temporal_triple(g, person, general.locatedAt, general[location], event_time)

# --- Drift Injection ---
def inject_drift(g, person, start_date):
    drift_day = random.randint(10, 20)
    d = start_date + timedelta(days=drift_day)
    # Health drift: new condition or increased visits
    simulate_doctor_visit(g, person, "DrBob", "CityClinic", ["Fatigue"], ["Diabetes"], d)
    # Routine drift: change activity/location
    simulate_routine_activity(g, person, "Yoga", "WellnessCenter", d)
    # Travel drift: simulate relocation
    add_temporal_triple(g, person, general.relocatedTo, general["NewCity"], d)

# --- Simulate for multiple people ---
people = [ex.JohnDoe, ex.JaneSmith, ex.Robert]
start_date = datetime(2025, 1, 1)

for i in range(60):  # simulate for 30 days
    for person in people:
        d = start_date + timedelta(days=i)

        if random.random() < 0.4:
            simulate_doctor_visit(
                g, person, "DrAlice", "CityClinic",
                symptoms=["Cough", "Fever"],
                diagnoses=["Flu"],
                start_date=d
            )

        if random.random() < 0.3:
            simulate_finance_event(
                g, person, "BankA", "MutualFundX", round(random.uniform(100, 500), 2), d
            )

        if random.random() < 0.5:
            simulate_routine_activity(
                g, person, "Workout", "GymPark", d
            )

# Inject temporal drift for each person
for person in people:
    inject_drift(g, person, start_date)



In [4]:
# Save
g.serialize("simulated_tkg.ttl", format="turtle")


<Graph identifier=N9a7a3a99493c4273bef60bb9018a8001 (<class 'rdflib.graph.Graph'>)>

In [5]:
len(g)

763

Simple case: we know the context

In [3]:
from rdflib import Graph, URIRef, Literal
from rdflib.namespace import XSD
from datetime import datetime, timedelta
from collections import defaultdict
import hashlib
import networkx as nx
from karateclub import Graph2Vec
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np



In [8]:
# === Config ===
HEALTH_NS = "http://example.org/health/"
TIME_PRED = URIRef("http://example.org/time/timestamp")

def get_timestamp_literal(triple_graph, stmt_id):
    for _, _, ts in triple_graph.triples((stmt_id, TIME_PRED, None)):
        return datetime.fromisoformat(str(ts))
    return None

def extract_context_subgraph(graph, person_uri, context="health", until_time=None):
    G = nx.DiGraph()
    context_prefix = f"http://example.org/{context}/"

    for s, p, o in graph:
        if not isinstance(o, URIRef):
            continue
        stmt_id = URIRef(f"http://example.org/event/{hashlib.md5(f'{s}_{p}_{o}'.encode()).hexdigest()}")
        ts = get_timestamp_literal(graph, stmt_id)
        if until_time and (not ts or ts > until_time):
            continue
        if str(s) != str(person_uri):
            continue
        if str(p).startswith(context_prefix) or str(o).startswith(context_prefix):
            G.add_edge(str(s), str(o), label=str(p).split("/")[-1])

    return G

def extract_future_snapshots(graph, person_uri, context="health", from_time=None, window_days=30, num_snapshots=6):
    snapshots = []
    for i in range(num_snapshots):
        until_time = from_time + timedelta(days=window_days * (i + 1))
        G = extract_context_subgraph(graph, person_uri, context=context, until_time=until_time)
        snapshots.append((until_time, G))
    return snapshots

# def encode_graphs(graphs):
#     model = Graph2Vec(dimensions=64, wl_iterations=2, min_count=1, epochs=10)
#     model.fit(graphs)
#     embeddings = model.get_embedding()
#     return embeddings

def encode_graphs(graphs):
    relabeled_graphs = []
    for G in graphs:
        mapping = {node: i for i, node in enumerate(G.nodes())}
        G_relabel = nx.relabel_nodes(G, mapping)
        relabeled_graphs.append(G_relabel)

    model = Graph2Vec(dimensions=64, wl_iterations=2, min_count=1, epochs=10)
    model.fit(relabeled_graphs)
    embeddings = model.get_embedding()
    return embeddings


def classify_evolution(reference_emb, future_embs):
    sims = [cosine_similarity([reference_emb], [emb])[0][0] for emb in future_embs]
    deltas = np.diff(sims)

    results = []
    for i, sim in enumerate(sims):
        if i == 0:
            trend = "baseline"
        elif deltas[i-1] > 0.1:
            trend = "growing"
        elif deltas[i-1] < -0.1:
            trend = "decaying"
        elif abs(deltas[i-1]) < 0.05:
            trend = "stable"
        else:
            trend = "drifting"
        results.append((i, sim, trend))
    return results

def run_health_evolution(graph, person_uri, context="health", window_days=30, num_snapshots=6):
    all_timestamps = [get_timestamp_literal(graph, URIRef(stmt)) for stmt in graph.subjects(predicate=TIME_PRED)]
    all_timestamps = sorted([ts for ts in all_timestamps if ts])
    if not all_timestamps:
        raise ValueError("No timestamps found.")

    final_time = all_timestamps[-1]
    T0 = final_time - timedelta(days=window_days * num_snapshots)

    G_ref = extract_context_subgraph(graph, person_uri, context, until_time=T0)
    future_snapshots = extract_future_snapshots(graph, person_uri, context, from_time=T0, window_days=window_days, num_snapshots=num_snapshots)

    graphs = [G_ref] + [G for _, G in future_snapshots]
    embeddings = encode_graphs(graphs)

    evolution = classify_evolution(embeddings[0], embeddings[1:])

    print("Subgraph evolution for:", person_uri.split("/")[-1])
    for i, sim, label in evolution:
        print(f" T{i+1}: similarity = {sim:.3f}, label = {label}")

    return evolution


In [9]:

ex = Namespace("http://example.org/")
run_health_evolution(g, person_uri=ex.JohnDoe)

Subgraph evolution for: JohnDoe
 T1: similarity = -0.031, label = baseline
 T2: similarity = 0.119, label = growing
 T3: similarity = -0.233, label = decaying
 T4: similarity = -0.108, label = growing
 T5: similarity = -0.034, label = drifting
 T6: similarity = 0.063, label = drifting


[(0, -0.030501723, 'baseline'),
 (1, 0.118750766, 'growing'),
 (2, -0.23339382, 'decaying'),
 (3, -0.10824738, 'growing'),
 (4, -0.03356202, 'drifting'),
 (5, 0.06270765, 'drifting')]

Generalized

In [12]:
from rdflib import Graph, URIRef, Literal
from rdflib.namespace import XSD
from datetime import datetime, timedelta
from collections import defaultdict
import hashlib
import networkx as nx
from karateclub import Graph2Vec
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# === Config ===
TIME_PRED = URIRef("http://example.org/time/timestamp")

# --- Utilities ---
def get_timestamp_literal(triple_graph, stmt_id):
    for _, _, ts in triple_graph.triples((stmt_id, TIME_PRED, None)):
        return datetime.fromisoformat(str(ts))
    return None

def extract_context_subgraph(graph, person_uri, context_prefix, until_time=None):
    G = nx.DiGraph()

    for s, p, o in graph:
        if not isinstance(o, URIRef):
            continue
        stmt_id = URIRef(f"http://example.org/event/{hashlib.md5(f'{s}_{p}_{o}'.encode()).hexdigest()}")
        ts = get_timestamp_literal(graph, stmt_id)
        if until_time and (not ts or ts > until_time):
            continue
        if str(s) != str(person_uri):
            continue
        if str(p).startswith(context_prefix) or str(o).startswith(context_prefix):
            G.add_edge(str(s), str(o), label=str(p).split("/")[-1])

    return G

def extract_future_snapshots(graph, person_uri, context_prefix, from_time=None, window_days=30, num_snapshots=6):
    snapshots = []
    for i in range(num_snapshots):
        until_time = from_time + timedelta(days=window_days * (i + 1))
        G = extract_context_subgraph(graph, person_uri, context_prefix, until_time=until_time)
        snapshots.append((until_time, G))
    return snapshots

def encode_graphs(graphs):
    relabeled_graphs = []
    for G in graphs:
        mapping = {node: i for i, node in enumerate(G.nodes())}
        G_relabel = nx.relabel_nodes(G, mapping)
        relabeled_graphs.append(G_relabel)

    model = Graph2Vec(dimensions=64, wl_iterations=2, min_count=1, epochs=10)
    model.fit(relabeled_graphs)
    embeddings = model.get_embedding()
    return embeddings

def classify_evolution(reference_emb, future_embs):
    sims = [cosine_similarity([reference_emb], [emb])[0][0] for emb in future_embs]
    deltas = np.diff(sims)

    results = []
    for i, sim in enumerate(sims):
        if i == 0:
            trend = "baseline"
        elif deltas[i-1] > 0.1:
            trend = "growing"
        elif deltas[i-1] < -0.1:
            trend = "decaying"
        elif abs(deltas[i-1]) < 0.05:
            trend = "stable"
        else:
            trend = "drifting"
        results.append((i, sim, trend))
    return results

def run_context_evolution(graph, person_uri, context="health", window_days=30, num_snapshots=6):
    context_prefix = f"http://example.org/{context}/"

    all_timestamps = [get_timestamp_literal(graph, URIRef(stmt)) for stmt in graph.subjects(predicate=TIME_PRED)]
    all_timestamps = sorted([ts for ts in all_timestamps if ts])
    if not all_timestamps:
        raise ValueError("No timestamps found.")

    final_time = all_timestamps[-1]
    T0 = final_time - timedelta(days=window_days * num_snapshots)

    G_ref = extract_context_subgraph(graph, person_uri, context_prefix, until_time=T0)
    future_snapshots = extract_future_snapshots(graph, person_uri, context_prefix, from_time=T0, window_days=window_days, num_snapshots=num_snapshots)

    graphs = [G_ref] + [G for _, G in future_snapshots]
    embeddings = encode_graphs(graphs)

    evolution = classify_evolution(embeddings[0], embeddings[1:])

    print(f"Subgraph evolution for: {person_uri.split('/')[-1]} (context: {context})")
    for i, sim, label in evolution:
        print(f" T{i+1}: similarity = {sim:.3f}, label = {label}")

    return evolution


In [14]:
# Detect drift in health domain for JohnDoe
run_context_evolution(graph=g, person_uri=ex.JohnDoe, context="finance")


Subgraph evolution for: JohnDoe (context: finance)
 T1: similarity = -0.031, label = baseline
 T2: similarity = 0.119, label = growing
 T3: similarity = -0.233, label = decaying
 T4: similarity = -0.108, label = growing
 T5: similarity = -0.034, label = drifting
 T6: similarity = 0.063, label = drifting


[(0, -0.030501723, 'baseline'),
 (1, 0.118750766, 'growing'),
 (2, -0.23339382, 'decaying'),
 (3, -0.10824738, 'growing'),
 (4, -0.033627324, 'drifting'),
 (5, 0.06274391, 'drifting')]

In [15]:
from rdflib import Graph, URIRef, Literal
from rdflib.namespace import XSD
from datetime import datetime, timedelta
from collections import defaultdict
import hashlib
import networkx as nx
from karateclub import Graph2Vec
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# === Config ===
TIME_PRED = URIRef("http://example.org/time/timestamp")

# --- Utilities ---
def get_timestamp_literal(triple_graph, stmt_id):
    for _, _, ts in triple_graph.triples((stmt_id, TIME_PRED, None)):
        return datetime.fromisoformat(str(ts))
    return None

def extract_context_subgraph(graph, person_uri, context_prefix, until_time=None):
    G = nx.DiGraph()

    for s, p, o in graph:
        if not isinstance(o, URIRef):
            continue
        stmt_id = URIRef(f"http://example.org/event/{hashlib.md5(f'{s}_{p}_{o}'.encode()).hexdigest()}")
        ts = get_timestamp_literal(graph, stmt_id)
        if until_time and (not ts or ts > until_time):
            continue
        if str(s) != str(person_uri):
            continue
        if str(p).startswith(context_prefix) or str(o).startswith(context_prefix):
            G.add_edge(str(s), str(o), label=str(p).split("/")[-1])

    return G

def extract_future_snapshots(graph, person_uri, context_prefix, from_time=None, window_days=30, num_snapshots=6):
    snapshots = []
    for i in range(num_snapshots):
        until_time = from_time + timedelta(days=window_days * (i + 1))
        G = extract_context_subgraph(graph, person_uri, context_prefix, until_time=until_time)
        snapshots.append((until_time, G))
    return snapshots

def encode_graphs(graphs):
    relabeled_graphs = []
    for G in graphs:
        mapping = {node: i for i, node in enumerate(G.nodes())}
        G_relabel = nx.relabel_nodes(G, mapping)
        relabeled_graphs.append(G_relabel)

    model = Graph2Vec(dimensions=64, wl_iterations=2, min_count=1, epochs=10)
    model.fit(relabeled_graphs)
    embeddings = model.get_embedding()
    return embeddings

def classify_evolution(reference_emb, future_embs):
    sims = [cosine_similarity([reference_emb], [emb])[0][0] for emb in future_embs]
    deltas = np.diff(sims)

    results = []
    for i, sim in enumerate(sims):
        if i == 0:
            trend = "baseline"
        elif deltas[i-1] > 0.1:
            trend = "growing"
        elif deltas[i-1] < -0.1:
            trend = "decaying"
        elif abs(deltas[i-1]) < 0.05:
            trend = "stable"
        else:
            trend = "drifting"
        results.append((i, sim, trend))
    return results

def run_context_evolution(graph, person_uri, context="health", window_days=30, num_snapshots=6):
    context_prefix = f"http://example.org/{context}/"
    context_prefix = f"http://example.org/{context}/"

    all_timestamps = [get_timestamp_literal(graph, URIRef(stmt)) for stmt in graph.subjects(predicate=TIME_PRED)]
    all_timestamps = sorted([ts for ts in all_timestamps if ts])
    if not all_timestamps:
        raise ValueError("No timestamps found.")

    final_time = all_timestamps[-1]
    T0 = final_time - timedelta(days=window_days * num_snapshots)

    G_ref = extract_context_subgraph(graph, person_uri, context_prefix, until_time=T0)
    print("Reference Subgraph (nodes, edges):", G_ref.number_of_nodes(), G_ref.number_of_edges())
    print("Triples:")
    for u, v, data in G_ref.edges(data=True):
        print(f"  ({u}, {data['label']}, {v})")
    future_snapshots = extract_future_snapshots(graph, person_uri, context_prefix, from_time=T0, window_days=window_days, num_snapshots=num_snapshots)

    graphs = [G_ref] + [G for _, G in future_snapshots]
    embeddings = encode_graphs(graphs)

    evolution = classify_evolution(embeddings[0], embeddings[1:])

    print(f"Subgraph evolution for: {person_uri.split('/')[-1]} (context: {context})")
    for i, sim, label in evolution:
        print(f" T{i+1}: similarity = {sim:.3f}, label = {label}")

    return evolution


In [23]:
# Detect drift in health domain for JohnDoe
# run_context_evolution(graph=g, person_uri=ex.JohnDoe, context="finance")
run_context_evolution(g, ex.JohnDoe, context="health", window_days=5, num_snapshots=10)


Reference Subgraph (nodes, edges): 6 5
Triples:
  (http://example.org/JohnDoe, atHospital, http://example.org/health/CityClinic)
  (http://example.org/JohnDoe, hasSymptom, http://example.org/health/Fever)
  (http://example.org/JohnDoe, visitedDoctor, http://example.org/health/DrAlice)
  (http://example.org/JohnDoe, hasSymptom, http://example.org/health/Cough)
  (http://example.org/JohnDoe, diagnosedWith, http://example.org/health/Flu)
Subgraph evolution for: JohnDoe (context: health)
 T1: similarity = -0.019, label = baseline
 T2: similarity = 0.122, label = growing
 T3: similarity = -0.223, label = decaying
 T4: similarity = -0.097, label = growing
 T5: similarity = -0.027, label = drifting
 T6: similarity = 0.074, label = growing
 T7: similarity = -0.040, label = decaying
 T8: similarity = 0.038, label = drifting
 T9: similarity = -0.149, label = decaying
 T10: similarity = -0.089, label = drifting


[(0, -0.019253474, 'baseline'),
 (1, 0.12156969, 'growing'),
 (2, -0.22303079, 'decaying'),
 (3, -0.09689755, 'growing'),
 (4, -0.026552811, 'drifting'),
 (5, 0.07403053, 'growing'),
 (6, -0.040130068, 'decaying'),
 (7, 0.03837431, 'drifting'),
 (8, -0.14855886, 'decaying'),
 (9, -0.08941409, 'drifting')]